# Wax strip cell counting
Upload a scan PDF -> strips are cut automatically -> filled cells are segmented (Cellpose) -> counts per strip.

Runtime -> Change runtime type -> **T4 GPU**, then run the cells top to bottom.

## 1. Upload PDF(s)
Click **Upload PDF(s)**, pick your files, wait for the file name to appear, then run the next cell.

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import output; output.enable_custom_widget_manager()
except Exception:
    pass
uploader = widgets.FileUpload(accept='.pdf', multiple=True, description='Upload PDF(s)', layout=widgets.Layout(width='220px'))
display(uploader)

In [ ]:
import glob
os.makedirs('/content/input', exist_ok=True)
val = uploader.value
# ipywidgets 7: {name: {content,...}}   ipywidgets 8: ({name, content, ...}, ...)
items = [(n, d) for n, d in val.items()] if isinstance(val, dict) else [(d['name'], d) for d in val]
for name, d in items:
    open(f'/content/input/{name}', 'wb').write(bytes(d['content']))
pdfs = sorted(glob.glob('/content/input/*.pdf'))
if not pdfs:   # fallback: Drive folder
    pdfs = sorted(glob.glob('/content/drive/MyDrive/wax_scans/*.pdf'))
print(len(pdfs), 'PDF(s):', [os.path.basename(p) for p in pdfs])
assert pdfs, 'No PDFs: upload one above (wait for the file name to appear) or put it in MyDrive/wax_scans'

## 2. Setup

In [ ]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU runtime')
!pip install -q cellpose==4.2.1.1 opencv-python-headless tifffile pymupdf

REPO = 'github.com/Mateo-Schaerer-Gonzalez/recapping_Analysis.git'   # public repo
os.chdir('/content')
if not os.path.exists('recapping_Analysis'):
    subprocess.run(['git', 'clone', f'https://{REPO}'], check=True)
else:
    subprocess.run(['git', '-C', 'recapping_Analysis', 'pull'], check=True)
os.chdir('/content/recapping_Analysis')
print(os.getcwd(), os.path.exists('wax_pipeline.py'))

IndentationError: unexpected indent (887162081.py, line 5)

## 3. Model
The fine-tuned model (`wax_cpsam`, ~1.2 GB) is not in git. Put it once in your Google Drive (e.g. `MyDrive/wax_cpsam`) and set the path below.
If the file is not found, the generic Cellpose model is used instead (it misses many cells on dense patches).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/wax_cpsam'
MODEL = MODEL_PATH if os.path.exists(MODEL_PATH) else None
print('using', MODEL or 'generic cpsam_v2 (fine-tuned model not found!)')

## 4. Run: strips -> segmentation -> counts

In [ ]:
import cv2, pandas as pd
import matplotlib.pyplot as plt
import wax_pipeline as wp

def show(name, overlay):
    plt.figure(figsize=(12, 6)); plt.imshow(overlay[..., ::-1]); plt.axis('off'); plt.title(name); plt.show()

results = []
for pdf in pdfs:
    stem = os.path.splitext(os.path.basename(pdf))[0]
    df = wp.run_pdf(pdf, out_dir=f'/content/results/{stem}', model=MODEL, show=show)
    df.insert(0, 'file', stem)
    results.append(df)
counts = pd.concat(results, ignore_index=True)
counts.to_csv('/content/results/all_counts.csv', index=False)
counts

## 5. Download results
Zip with per-strip images, label masks (`.tif`), overlays and `counts.csv`.

In [ ]:
from google.colab import files
!cd /content && zip -qr results.zip results
files.download('/content/results.zip')
# also saved to Drive so you can fetch it later:
!cp /content/results.zip /content/drive/MyDrive/wax_results.zip